# Tutorial: Ejercicio 3.3 en AMPL - inventario de mascarillas for dummies

Audiencia:
- Personas que ya entendieron la idea del problema y ahora quieren verlo en AMPL.
- Personas que quieren aprender como se representa un modelo de optimizacion con sets, parametros, variables, objetivo y restricciones.

Prerrequisitos:
- Entender la historia del problema de inventario.
- Saber leer Python basico.

Objetivos:
- Ver todas las decisiones del modelo.
- Traducir el problema a AMPL paso a paso.
- Resolver el modelo y aprender a leer la solucion.


## Enunciado del ejercicio

Una empresa distribuidora de tapabocas tiene demanda pronosticada para los proximos cuatro meses de `2800`, `2200`, `3200` y `2500` unidades para los meses `1`, `2`, `3` y `4`, respectivamente.

Datos del enunciado:
- capacidad de produccion normal: `2700` unidades por mes,
- capacidad adicional en tiempo extra: `300` unidades por mes,
- sobrecosto de tiempo extra: `10` USD por unidad,
- costo de almacenamiento: `2` USD por unidad que quede guardada al final de un mes.

Se asume inventario inicial `0` y se desea inventario final `0`.

Pregunta:
**¿Cual es el programa optimo de produccion que minimiza los costos de tiempo extra y almacenamiento, cumpliendo la demanda de cada mes?**


## Mapa del notebook

1. Ver los datos.
2. Entender como piensa AMPL.
3. Identificar decisiones, objetivo y restricciones.
4. Escribir el modelo AMPL.
5. Resolverlo con `amplpy`.
6. Leer la solucion y compararla con la intuicion.


In [1]:
MESES = [1, 2, 3, 4]
MESES_CON_CERO = [0, 1, 2, 3, 4]
DEMANDA = {1: 2800, 2: 2200, 3: 3200, 4: 2500}
CAP_NORMAL = 2700
CAP_EXTRA = 300
COSTO_EXTRA = 10
COSTO_INVENTARIO = 2
ESPERADO = {
    "normal": {1: 2700, 2: 2700, 3: 2700, 4: 2500},
    "extra": {1: 100, 2: 0, 3: 0, 4: 0},
    "inventario": {0: 0, 1: 0, 2: 500, 3: 0, 4: 0},
    "costo": 2000,
}

{
    "meses": MESES,
    "demanda": DEMANDA,
    "capacidad_normal": CAP_NORMAL,
    "capacidad_extra": CAP_EXTRA,
    "costo_extra": COSTO_EXTRA,
    "costo_inventario": COSTO_INVENTARIO,
}


{'meses': [1, 2, 3, 4],
 'demanda': {1: 2800, 2: 2200, 3: 3200, 4: 2500},
 'capacidad_normal': 2700,
 'capacidad_extra': 300,
 'costo_extra': 10,
 'costo_inventario': 2}

## Paso 0 - Como piensa AMPL

AMPL separa el modelo en piezas.

Las piezas principales son:
- `set`: conjuntos de indices. Aqui, los meses.
- `param`: datos conocidos. Aqui, demanda, capacidades y costos.
- `var`: variables de decision. Aqui, produccion normal, produccion extra e inventario.
- `minimize`: la funcion objetivo.
- `subject to`: las restricciones.

Si piensas el problema con esas cinco piezas, AMPL se vuelve mucho menos misterioso.


In [2]:
traduccion_a_ampl = [
    {"idea": "meses", "en_python": MESES, "en_ampl": "set MESES;"},
    {"idea": "demanda", "en_python": DEMANDA, "en_ampl": "param demanda {MESES};"},
    {"idea": "produccion normal", "en_python": "decision por mes", "en_ampl": "var ProduccionNormal {MESES} >= 0;"},
    {"idea": "produccion extra", "en_python": "decision por mes", "en_ampl": "var ProduccionExtra {MESES} >= 0;"},
    {"idea": "inventario", "en_python": "decision por mes", "en_ampl": "var Inventario {MESES_CON_CERO} >= 0;"},
]

traduccion_a_ampl


[{'idea': 'meses', 'en_python': [1, 2, 3, 4], 'en_ampl': 'set MESES;'},
 {'idea': 'demanda',
  'en_python': {1: 2800, 2: 2200, 3: 3200, 4: 2500},
  'en_ampl': 'param demanda {MESES};'},
 {'idea': 'produccion normal',
  'en_python': 'decision por mes',
  'en_ampl': 'var ProduccionNormal {MESES} >= 0;'},
 {'idea': 'produccion extra',
  'en_python': 'decision por mes',
  'en_ampl': 'var ProduccionExtra {MESES} >= 0;'},
 {'idea': 'inventario',
  'en_python': 'decision por mes',
  'en_ampl': 'var Inventario {MESES_CON_CERO} >= 0;'}]

## Paso 1 - Todas las decisiones del modelo

En la version AMPL no simplificamos las decisiones.
Aqui el modelo decide directamente:
- cuanto producir normal en cada mes,
- cuanto producir extra en cada mes,
- cuanto inventario dejar al final de cada mes.

Eso es importante: en AMPL escribimos el problema completo, no solo la version reducida pedagogica.


In [3]:
variables_de_decision = [
    {
        "variable": "ProduccionNormal[t]",
        "significado": "unidades hechas en horario normal durante el mes t",
        "tipo": "entera y no negativa",
    },
    {
        "variable": "ProduccionExtra[t]",
        "significado": "unidades hechas en horario extra durante el mes t",
        "tipo": "entera y no negativa",
    },
    {
        "variable": "Inventario[t]",
        "significado": "unidades guardadas al final del mes t",
        "tipo": "entera y no negativa",
    },
]

variables_de_decision


[{'variable': 'ProduccionNormal[t]',
  'significado': 'unidades hechas en horario normal durante el mes t',
  'tipo': 'entera y no negativa'},
 {'variable': 'ProduccionExtra[t]',
  'significado': 'unidades hechas en horario extra durante el mes t',
  'tipo': 'entera y no negativa'},
 {'variable': 'Inventario[t]',
  'significado': 'unidades guardadas al final del mes t',
  'tipo': 'entera y no negativa'}]

## Paso 2 - Representacion del modelo en palabras

Objetivo:
- minimizar costo de produccion extra + costo de inventario.

Restricciones:
- la produccion normal no puede superar `2700` por mes,
- la produccion extra no puede superar `300` por mes,
- el balance de inventario debe cumplirse cada mes,
- se parte con inventario 0,
- se termina con inventario 0.

La restriccion de balance es la mas importante:

`inventario anterior + produccion normal + produccion extra = demanda + inventario final`


In [4]:
modelo_en_palabras = {
    "objetivo": "minimizar costo de produccion extra y costo de inventario",
    "restricciones": [
        "ProduccionNormal[t] <= cap_normal",
        "ProduccionExtra[t] <= cap_extra",
        "Inventario[t-1] + ProduccionNormal[t] + ProduccionExtra[t] = demanda[t] + Inventario[t]",
        "Inventario[0] = 0",
        "Inventario[4] = 0",
    ],
}

modelo_en_palabras


{'objetivo': 'minimizar costo de produccion extra y costo de inventario',
 'restricciones': ['ProduccionNormal[t] <= cap_normal',
  'ProduccionExtra[t] <= cap_extra',
  'Inventario[t-1] + ProduccionNormal[t] + ProduccionExtra[t] = demanda[t] + Inventario[t]',
  'Inventario[0] = 0',
  'Inventario[4] = 0']}

## Paso 3 - Preparar AMPL desde Python

Si en tu kernel aun no esta instalado `amplpy`, primero ejecuta esto en una celda aparte:

```python
%pip install amplpy
```

En este notebook asumimos que `amplpy` ya esta disponible.


In [5]:
from amplpy import ampl_notebook


def crear_ampl():
    ampl = ampl_notebook(modules=["highs"], license_uuid="default")
    ampl.option["solver"] = "highs"
    ampl.option["solver_msg"] = 0
    return ampl


def leer_variable_1d(variable, claves):
    bruto = variable.get_values().to_dict()
    limpio = {}
    for clave in claves:
        if clave in bruto:
            valor = bruto[clave]
        else:
            valor = bruto[(clave,)]
        limpio[clave] = int(round(float(valor)))
    return limpio


MODELO_AMPL = r'''
set MESES ordered;
set MESES_CON_CERO ordered;

param demanda {MESES} >= 0;
param cap_normal >= 0;
param cap_extra >= 0;
param costo_extra >= 0;
param costo_inventario >= 0;

# Variables de decision
var ProduccionNormal {t in MESES} integer >= 0;
var ProduccionExtra {t in MESES} integer >= 0;
var Inventario {t in MESES_CON_CERO} integer >= 0;

# Funcion objetivo
minimize CostoTotal:
    sum {t in MESES} costo_extra * ProduccionExtra[t]
  + sum {t in MESES} costo_inventario * Inventario[t];

# Restricciones de capacidad
subject to LimiteNormal {t in MESES}:
    ProduccionNormal[t] <= cap_normal;

subject to LimiteExtra {t in MESES}:
    ProduccionExtra[t] <= cap_extra;

# Restriccion de balance
subject to Balance {t in MESES}:
    Inventario[t - 1] + ProduccionNormal[t] + ProduccionExtra[t]
    = demanda[t] + Inventario[t];

# Condiciones de inicio y termino
subject to InventarioInicial:
    Inventario[0] = 0;

subject to InventarioFinal:
    Inventario[last(MESES)] = 0;
'''


## Paso 4 - Este es el modelo AMPL completo

Miremos el modelo entero antes de resolverlo.
La idea es que puedas reconocer visualmente donde estan:
- los datos,
- las variables,
- el objetivo,
- y las restricciones.


In [6]:
print(MODELO_AMPL)



set MESES ordered;
set MESES_CON_CERO ordered;

param demanda {MESES} >= 0;
param cap_normal >= 0;
param cap_extra >= 0;
param costo_extra >= 0;
param costo_inventario >= 0;

# Variables de decision
var ProduccionNormal {t in MESES} integer >= 0;
var ProduccionExtra {t in MESES} integer >= 0;
var Inventario {t in MESES_CON_CERO} integer >= 0;

# Funcion objetivo
minimize CostoTotal:
    sum {t in MESES} costo_extra * ProduccionExtra[t]
  + sum {t in MESES} costo_inventario * Inventario[t];

# Restricciones de capacidad
subject to LimiteNormal {t in MESES}:
    ProduccionNormal[t] <= cap_normal;

subject to LimiteExtra {t in MESES}:
    ProduccionExtra[t] <= cap_extra;

# Restriccion de balance
subject to Balance {t in MESES}:
    Inventario[t - 1] + ProduccionNormal[t] + ProduccionExtra[t]
    = demanda[t] + Inventario[t];

# Condiciones de inicio y termino
subject to InventarioInicial:
    Inventario[0] = 0;

subject to InventarioFinal:
    Inventario[last(MESES)] = 0;



## Paso 5 - Resolver el modelo y leer la salida

Aqui hacemos cuatro cosas:
- crear la instancia de AMPL,
- cargar el modelo,
- pasarle los datos,
- resolver y traer la solucion de vuelta a Python.


In [7]:
def resolver_modelo_ampl(cap_normal=CAP_NORMAL, cap_extra=CAP_EXTRA, costo_extra=COSTO_EXTRA, costo_inventario=COSTO_INVENTARIO):
    ampl = crear_ampl()
    ampl.eval(MODELO_AMPL)

    ampl.set["MESES"] = MESES
    ampl.set["MESES_CON_CERO"] = MESES_CON_CERO
    ampl.param["demanda"] = DEMANDA
    ampl.param["cap_normal"] = cap_normal
    ampl.param["cap_extra"] = cap_extra
    ampl.param["costo_extra"] = costo_extra
    ampl.param["costo_inventario"] = costo_inventario

    ampl.solve()

    resultado = {
        "normal": leer_variable_1d(ampl.var["ProduccionNormal"], MESES),
        "extra": leer_variable_1d(ampl.var["ProduccionExtra"], MESES),
        "inventario": leer_variable_1d(ampl.var["Inventario"], MESES_CON_CERO),
        "costo": int(round(float(ampl.obj["CostoTotal"].value()))),
    }

    return ampl, resultado


ampl_base, resultado_ampl = resolver_modelo_ampl()

print("Resultado en Python:")
print(resultado_ampl)
print()
print("Salida nativa de AMPL:")
print(ampl_base.get_output("display ProduccionNormal, ProduccionExtra, Inventario;"))

assert resultado_ampl == ESPERADO

resumen_ampl = []
for mes in MESES:
    resumen_ampl.append(
        {
            "mes": mes,
            "normal": resultado_ampl["normal"][mes],
            "extra": resultado_ampl["extra"][mes],
            "inventario_final": resultado_ampl["inventario"][mes],
        }
    )

resumen_ampl


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: Resultado en Python:
{'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500}, 'extra': {1: 100, 2: 0, 3: 0, 4: 0}, 'inventario': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}, 'costo': 2000}

Salida nativa de AMPL:
: ProduccionNormal ProduccionExtra Inventario    :=
0          .               .             0
1        2700             100            0
2        2700               0          500
3        2700               0            0
4        2500               0            0
;




[{'mes': 1, 'normal': 2700, 'extra': 100, 'inventario_final': 0},
 {'mes': 2, 'normal': 2700, 'extra': 0, 'inventario_final': 500},
 {'mes': 3, 'normal': 2700, 'extra': 0, 'inventario_final': 0},
 {'mes': 4, 'normal': 2500, 'extra': 0, 'inventario_final': 0}]

## Paso 6 - Como leer la solucion

La solucion de AMPL cuenta la misma historia que la version Python:
- mes 1: un poco de produccion extra,
- mes 2: se guarda inventario,
- mes 3: ese inventario ayuda a cubrir la demanda alta,
- mes 4: se termina limpio, sin stock.

Lo importante es esto:
**AMPL no cambia la logica del problema. Solo nos deja escribirla de forma mas formal y escalable.**


## Ejercicio guiado

Prueba ahora la misma pregunta de la version Python:
- que pasa si la capacidad normal sube a `2800`?
- baja el costo?
- desaparece la produccion extra?


In [8]:
_, escenario_ampl_mas_facil = resolver_modelo_ampl(cap_normal=2800)

{
    "costo_original": resultado_ampl["costo"],
    "costo_con_capacidad_normal_2800": escenario_ampl_mas_facil["costo"],
    "extra_con_capacidad_normal_2800": escenario_ampl_mas_facil["extra"],
    "inventario_con_capacidad_normal_2800": escenario_ampl_mas_facil["inventario"],
}


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: 

{'costo_original': 2000,
 'costo_con_capacidad_normal_2800': 800,
 'extra_con_capacidad_normal_2800': {1: 0, 2: 0, 3: 0, 4: 0},
 'inventario_con_capacidad_normal_2800': {0: 0, 1: 0, 2: 400, 3: 0, 4: 0}}